# Osteoporosis Classification using Deep Learning

## Custom CNN Model - With Diifferent Droupts , Epochs and Learning Rate - Training Set Tabe

### Epochs = 10, Droput out = [0.0, 0.2, 0.4], Batch Size = [2, 4, 8, 16]

In [ ]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.regularizers import l2
from sklearn.metrics import precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix

# Load CSV file
csv_path = "/kaggle/working/Osteoporosis.csv"
df = pd.read_csv(csv_path)

# Encode labels
label_mapping = {label: idx for idx, label in enumerate(df['label'].unique())}
df['label'] = df['label'].map(label_mapping)

# Load and augment images using Pillow
def load_image(image_path, target_size=(224, 224)):
    img = Image.open(image_path).convert('L').resize(target_size)
    img = img_to_array(img) / 255.0  # Normalize
    img = np.expand_dims(img, axis=-1)  # Add channel dimension (H, W) → (H, W, 1)
    return img

image_paths = df['image'].values
labels = df['label'].values

# Split dataset (80% training, 20% validation)
train_paths, val_paths, train_labels, val_labels = train_test_split(image_paths, labels, test_size=0.2, stratify=labels)

# Data generator with augmentation
data_gen = ImageDataGenerator(rescale=1./255, rotation_range=20, zoom_range=0.2, horizontal_flip=True)

def data_generator(image_paths, labels, batch_size=64):
    while True:
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            images = np.array([load_image(img_path) for img_path in batch_paths])
            yield images, tf.keras.utils.to_categorical(batch_labels, num_classes=3)

train_gen = data_generator(train_paths, train_labels)
val_gen = data_generator(val_paths, val_labels)

# Function to create Custom CNN model
def create_custom_cnn(dropout_rate=0.0):
    input_layer = Input(shape=(224, 224, 3))

    x = Conv2D(128, (8, 8), activation='relu', padding='same')(input_layer)
    x = Conv2D(256, (5, 5), activation='relu', padding='same')(x)
    x = MaxPooling2D((3, 3))(x)

    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = Conv2D(256, (1, 1), activation='relu', padding='same')(x)

    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)

    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)

    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)

    x = Flatten()(x)
    
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)

    output = Dense(3, activation='softmax')(x)

    model = Model(inputs=input_layer, outputs=output)
    return model


def compute_metrics(true_labels, pred_labels):
    precision = precision_score(true_labels, pred_labels, average='weighted')
    recall = recall_score(true_labels, pred_labels, average='weighted')
    f1 = f1_score(true_labels, pred_labels, average='weighted')
    auc = roc_auc_score(tf.keras.utils.to_categorical(true_labels, num_classes=3),
                        tf.keras.utils.to_categorical(pred_labels, num_classes=3),
                        multi_class='ovr')
    
    cm = confusion_matrix(true_labels, pred_labels)
    specificity = np.mean([
        cm[i][i] / (sum(cm[:, i]) if sum(cm[:, i]) > 0 else 1)
        for i in range(len(cm))
    ])
    sensitivity = np.mean([
        cm[i][i] / (sum(cm[i]) if sum(cm[i]) > 0 else 1)
        for i in range(len(cm))
    ])
    
    return precision, recall, auc, f1, specificity, sensitivity


# Train and Evaluate the Custom CNN Model
results = []

for epochs in [10]:
    for dropout in [0.0, 0.2, 0.5]:
        for batch_size in [2, 4, 8, 16]:
            print(f"\nTraining: Epochs={epochs}, Dropout={dropout}, Batch Size={batch_size}")
            
            model = create_custom_cnn(dropout_rate=dropout)
            model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

            train_gen = data_generator(train_paths, train_labels, batch_size=batch_size)
            val_gen = data_generator(val_paths, val_labels, batch_size=batch_size)
            
            history = model.fit(
                train_gen,
                validation_data=val_gen,
                steps_per_epoch=len(train_paths) // batch_size,
                validation_steps=len(val_paths) // batch_size,
                epochs=epochs,
                verbose=0
            )

            val_images = np.array([load_image(p) for p in val_paths])
            y_true = val_labels
            y_pred = np.argmax(model.predict(val_images), axis=1)

            loss, acc = model.evaluate(val_images, tf.keras.utils.to_categorical(val_labels, num_classes=3), verbose=0)
            precision, recall, auc, f1, specificity, sensitivity = compute_metrics(y_true, y_pred)

            results.append({
                "Epochs": epochs,
                "Dropout": "Without" if dropout == 0 else dropout,
                "Batch Size": batch_size,
                "Loss": round(loss, 4),
                "Accuracy": round(acc, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "AUC": round(auc, 4),
                "F1 Score": round(f1, 4),
                "Specificity": round(specificity, 4),
                "Sensitivity": round(sensitivity, 4)
            })


results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["Epochs", "Dropout", "Batch Size"])
results_df.to_csv("/kaggle/working/custom_cnn_results.csv", index=False)
results_df


Training: Epochs=10, Dropout=0.0, Batch Size=2
13/13 ━━━━━━━━━━━━━━━━━━━━ 55s 1s/step 

Training: Epochs=10, Dropout=0.0, Batch Size=4
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 280ms/step

Training: Epochs=10, Dropout=0.0, Batch Size=8
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 292ms/step

Training: Epochs=10, Dropout=0.0, Batch Size=16
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 268ms/step

Training: Epochs=10, Dropout=0.2, Batch Size=2
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 279ms/step

Training: Epochs=10, Dropout=0.2, Batch Size=4
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 279ms/step

Training: Epochs=10, Dropout=0.2, Batch Size=8
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 277ms/step

Training: Epochs=10, Dropout=0.2, Batch Size=16
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 285ms/step

Training: Epochs=10, Dropout=0.5, Batch Size=2
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 279ms/step

Training: Epochs=10, Dropout=0.5, Batch Size=4
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 276ms/step

Training: Epochs=10, Dropout=0.5, Batch Size=8
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 282ms/step

Training: Epochs=10

,Epochs,Dropout,Batch Size,Loss,Accuracy,Precision,Recall,AUC,F1 Score,Specificity,Sensitivity
4,10,0.2,2,1.0504,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
5,10,0.2,4,1.0511,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
6,10,0.2,8,1.0501,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
7,10,0.2,16,0.4573,0.8590,0.8684,0.8590,0.8784,0.8578,0.8797,0.8343
8,10,0.5,2,1.0504,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
9,10,0.5,4,1.0509,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
10,10,0.5,8,1.0511,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
11,10,0.5,16,1.0506,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
0,10,Without,2,1.0505,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
1,10,Without,4,1.0506,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333


### Epochs = 15, Droput out = [0.0, 0.2, 0.5], Batch Size = [2, 4, 8, 16]

In [ ]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.regularizers import l2
from sklearn.metrics import precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix

# Load CSV file
csv_path = "/kaggle/working/Osteoporosis.csv"
df = pd.read_csv(csv_path)

# Encode labels
label_mapping = {label: idx for idx, label in enumerate(df['label'].unique())}
df['label'] = df['label'].map(label_mapping)

# Load and augment images using Pillow
def load_image(image_path, target_size=(224, 224)):
    img = Image.open(image_path).convert('L').resize(target_size)
    img = img_to_array(img) / 255.0  # Normalize
    img = np.expand_dims(img, axis=-1)  # Add channel dimension (H, W) → (H, W, 1)
    return img

image_paths = df['image'].values
labels = df['label'].values

# Split dataset (80% training, 20% validation)
train_paths, val_paths, train_labels, val_labels = train_test_split(image_paths, labels, test_size=0.2, stratify=labels)

# Data generator with augmentation
data_gen = ImageDataGenerator(rescale=1./255, rotation_range=20, zoom_range=0.2, horizontal_flip=True)

def data_generator(image_paths, labels, batch_size=64):
    while True:
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            images = np.array([load_image(img_path) for img_path in batch_paths])
            yield images, tf.keras.utils.to_categorical(batch_labels, num_classes=3)

train_gen = data_generator(train_paths, train_labels)
val_gen = data_generator(val_paths, val_labels)

# Function to create Custom CNN model
def create_custom_cnn(dropout_rate=0.0):
    input_layer = Input(shape=(224, 224, 3))

    x = Conv2D(128, (8, 8), activation='relu', padding='same')(input_layer)
    x = Conv2D(256, (5, 5), activation='relu', padding='same')(x)
    x = MaxPooling2D((3, 3))(x)

    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = Conv2D(256, (1, 1), activation='relu', padding='same')(x)

    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)

    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)

    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)

    x = Flatten()(x)
    
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)

    output = Dense(3, activation='softmax')(x)

    model = Model(inputs=input_layer, outputs=output)
    return model


def compute_metrics(true_labels, pred_labels):
    precision = precision_score(true_labels, pred_labels, average='weighted')
    recall = recall_score(true_labels, pred_labels, average='weighted')
    f1 = f1_score(true_labels, pred_labels, average='weighted')
    auc = roc_auc_score(tf.keras.utils.to_categorical(true_labels, num_classes=3),
                        tf.keras.utils.to_categorical(pred_labels, num_classes=3),
                        multi_class='ovr')
    
    cm = confusion_matrix(true_labels, pred_labels)
    specificity = np.mean([
        cm[i][i] / (sum(cm[:, i]) if sum(cm[:, i]) > 0 else 1)
        for i in range(len(cm))
    ])
    sensitivity = np.mean([
        cm[i][i] / (sum(cm[i]) if sum(cm[i]) > 0 else 1)
        for i in range(len(cm))
    ])
    
    return precision, recall, auc, f1, specificity, sensitivity


# Train and Evaluate the Custom CNN Model
results = []

for epochs in [15]:
    for dropout in [0.0, 0.2, 0.5]:
        for batch_size in [2, 4, 8, 16]:
            print(f"\nTraining: Epochs={epochs}, Dropout={dropout}, Batch Size={batch_size}")
            
            model = create_custom_cnn(dropout_rate=dropout)
            model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

            train_gen = data_generator(train_paths, train_labels, batch_size=batch_size)
            val_gen = data_generator(val_paths, val_labels, batch_size=batch_size)
            
            history = model.fit(
                train_gen,
                validation_data=val_gen,
                steps_per_epoch=len(train_paths) // batch_size,
                validation_steps=len(val_paths) // batch_size,
                epochs=epochs,
                verbose=0
            )

            val_images = np.array([load_image(p) for p in val_paths])
            y_true = val_labels
            y_pred = np.argmax(model.predict(val_images), axis=1)

            loss, acc = model.evaluate(val_images, tf.keras.utils.to_categorical(val_labels, num_classes=3), verbose=0)
            precision, recall, auc, f1, specificity, sensitivity = compute_metrics(y_true, y_pred)

            results.append({
                "Epochs": epochs,
                "Dropout": "Without" if dropout == 0 else dropout,
                "Batch Size": batch_size,
                "Loss": round(loss, 4),
                "Accuracy": round(acc, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "AUC": round(auc, 4),
                "F1 Score": round(f1, 4),
                "Specificity": round(specificity, 4),
                "Sensitivity": round(sensitivity, 4)
            })


results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["Epochs", "Dropout", "Batch Size"])
results_df.to_csv("/kaggle/working/custom_cnn_results.csv", index=False)
results_df


Training: Epochs=15, Dropout=0.0, Batch Size=2
13/13 ━━━━━━━━━━━━━━━━━━━━ 59s 1s/step 

Training: Epochs=15, Dropout=0.0, Batch Size=4
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 286ms/step

Training: Epochs=15, Dropout=0.0, Batch Size=8
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 289ms/step

Training: Epochs=15, Dropout=0.0, Batch Size=16
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 308ms/step

Training: Epochs=15, Dropout=0.2, Batch Size=2
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 287ms/step

Training: Epochs=15, Dropout=0.2, Batch Size=4
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 293ms/step

Training: Epochs=15, Dropout=0.2, Batch Size=8
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 286ms/step

Training: Epochs=15, Dropout=0.2, Batch Size=16
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 282ms/step

Training: Epochs=15, Dropout=0.5, Batch Size=2
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 291ms/step

Training: Epochs=15, Dropout=0.5, Batch Size=4
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 295ms/step

Training: Epochs=15, Dropout=0.5, Batch Size=8
13/13 ━━━━━━━━━━━━━━━━━━━━ 4s 291ms/step

Training: Epochs=15

,Epochs,Dropout,Batch Size,Loss,Accuracy,Precision,Recall,AUC,F1 Score,Specificity,Sensitivity
4,15,0.2,2,1.0502,0.4000,0.1600,0.4000,0.5000,0.2286,0.1333,0.3333
5,15,0.2,4,1.0497,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
6,15,0.2,8,1.0501,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
7,15,0.2,16,1.0520,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
8,15,0.5,2,1.0503,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
9,15,0.5,4,1.0495,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
10,15,0.5,8,1.0497,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
11,15,0.5,16,1.0531,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333
0,15,Without,2,1.0508,0.4000,0.1600,0.4000,0.5000,0.2286,0.1333,0.3333
1,15,Without,4,1.0495,0.4077,0.1662,0.4077,0.5000,0.2361,0.1359,0.3333


### Epochs = 20, Droput out = [0.0, 0.2, 0.4], Batch Size = [2, 4, 8, 16]

In [ ]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.model_selection import train_test_split
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization, LeakyReLU
from tensorflow.keras.regularizers import l2
from sklearn.metrics import precision_score, recall_score, roc_auc_score, f1_score, confusion_matrix

# Load CSV file
csv_path = "/kaggle/working/Osteoporosis.csv"
df = pd.read_csv(csv_path)

# Encode labels
label_mapping = {label: idx for idx, label in enumerate(df['label'].unique())}
df['label'] = df['label'].map(label_mapping)

# Load and augment images using Pillow
def load_image(image_path, target_size=(224, 224)):
    img = Image.open(image_path).convert('L').resize(target_size)
    img = img_to_array(img) / 255.0  # Normalize
    img = np.expand_dims(img, axis=-1)  # Add channel dimension (H, W) → (H, W, 1)
    return img

image_paths = df['image'].values
labels = df['label'].values

# Split dataset (80% training, 20% validation)
train_paths, val_paths, train_labels, val_labels = train_test_split(image_paths, labels, test_size=0.2, stratify=labels)

# Data generator with augmentation
data_gen = ImageDataGenerator(rescale=1./255, rotation_range=20, zoom_range=0.2, horizontal_flip=True)

def data_generator(image_paths, labels, batch_size=64):
    while True:
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i+batch_size]
            batch_labels = labels[i:i+batch_size]
            images = np.array([load_image(img_path) for img_path in batch_paths])
            yield images, tf.keras.utils.to_categorical(batch_labels, num_classes=3)

train_gen = data_generator(train_paths, train_labels)
val_gen = data_generator(val_paths, val_labels)

# Function to create Custom CNN model
def create_custom_cnn(dropout_rate=0.0):
    input_layer = Input(shape=(224, 224, 3))

    x = Conv2D(128, (8, 8), activation='relu', padding='same')(input_layer)
    x = Conv2D(256, (5, 5), activation='relu', padding='same')(x)
    x = MaxPooling2D((3, 3))(x)

    x = Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = Conv2D(256, (1, 1), activation='relu', padding='same')(x)

    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)

    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)

    x = Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    x = MaxPooling2D((2, 2))(x)

    x = Flatten()(x)
    
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)

    output = Dense(3, activation='softmax')(x)

    model = Model(inputs=input_layer, outputs=output)
    return model


def compute_metrics(true_labels, pred_labels):
    precision = precision_score(true_labels, pred_labels, average='weighted')
    recall = recall_score(true_labels, pred_labels, average='weighted')
    f1 = f1_score(true_labels, pred_labels, average='weighted')
    auc = roc_auc_score(tf.keras.utils.to_categorical(true_labels, num_classes=3),
                        tf.keras.utils.to_categorical(pred_labels, num_classes=3),
                        multi_class='ovr')
    
    cm = confusion_matrix(true_labels, pred_labels)
    specificity = np.mean([
        cm[i][i] / (sum(cm[:, i]) if sum(cm[:, i]) > 0 else 1)
        for i in range(len(cm))
    ])
    sensitivity = np.mean([
        cm[i][i] / (sum(cm[i]) if sum(cm[i]) > 0 else 1)
        for i in range(len(cm))
    ])
    
    return precision, recall, auc, f1, specificity, sensitivity


# Train and Evaluate the Custom CNN Model
results = []

for epochs in [20]:
    for dropout in [0.0, 0.2, 0.5]:
        for batch_size in [2, 4, 8, 16]:
            print(f"\nTraining: Epochs={epochs}, Dropout={dropout}, Batch Size={batch_size}")
            
            model = create_custom_cnn(dropout_rate=dropout)
            model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

            train_gen = data_generator(train_paths, train_labels, batch_size=batch_size)
            val_gen = data_generator(val_paths, val_labels, batch_size=batch_size)
            
            history = model.fit(
                train_gen,
                validation_data=val_gen,
                steps_per_epoch=len(train_paths) // batch_size,
                validation_steps=len(val_paths) // batch_size,
                epochs=epochs,
                verbose=0
            )

            val_images = np.array([load_image(p) for p in val_paths])
            y_true = val_labels
            y_pred = np.argmax(model.predict(val_images), axis=1)

            loss, acc = model.evaluate(val_images, tf.keras.utils.to_categorical(val_labels, num_classes=3), verbose=0)
            precision, recall, auc, f1, specificity, sensitivity = compute_metrics(y_true, y_pred)

            results.append({
                "Epochs": epochs,
                "Dropout": "Without" if dropout == 0 else dropout,
                "Batch Size": batch_size,
                "Loss": round(loss, 4),
                "Accuracy": round(acc, 4),
                "Precision": round(precision, 4),
                "Recall": round(recall, 4),
                "AUC": round(auc, 4),
                "F1 Score": round(f1, 4),
                "Specificity": round(specificity, 4),
                "Sensitivity": round(sensitivity, 4)
            })


results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by=["Epochs", "Dropout", "Batch Size"])
results_df.to_csv("/kaggle/working/custom_cnn_results.csv", index=False)
results_df